# Anomaly Detection (Isolation Forest & LOF)
**Repositori**: Machine Learning
**Topik**: Implementasi anomaly detection untuk deteksi fraud transaksi
**Dataset**: transaction_data.csv
---
**Pendahuluan**: Anomaly detection digunakan untuk mengidentifikasi pola langka yang mencurigakan. Isolation Forest dan Local Outlier Factor (LOF) adalah dua algoritma unsupervised yang populer.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)


## 2. Load & EDA


In [ ]:
df = pd.read_csv('../../data/transaction_data.csv')
print('Shape:', df.shape)
print(df.head())
print(df['Fraud Flag'].value_counts())
plt.figure(figsize=(6, 4))
sns.countplot(x='Fraud Flag', data=df)
plt.title('Distribusi Fraud Flag')
plt.show()


## 3. Data Preparation


In [ ]:
cols_to_drop = ['Transaction ID', 'Sender Account ID', 'Receiver Account ID', 'Timestamp', 'Geolocation (Latitude/Longitude)']
df_clean = df.drop(cols_to_drop, axis=1)
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = pd.factorize(df_clean[col])[0]
X = df_clean.drop('Fraud Flag', axis=1)
y = df_clean['Fraud Flag']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Fitur digunakan:', X.columns.tolist())
print(f'Jumlah sample: {X.shape[0]}, Fitur: {X.shape[1]}')


## 4. Isolation Forest


In [ ]:
iso_forest = IsolationForest(contamination=0.1, random_state=42)
y_pred_if = iso_forest.fit_predict(X_scaled)
y_pred_if = (y_pred_if == -1).astype(int)
print('=== Isolation Forest ===')
print(classification_report(y, y_pred_if, target_names=['Normal', 'Anomaly']))
cm = confusion_matrix(y, y_pred_if)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.title('Confusion Matrix - Isolation Forest')
plt.show()


## 5. Local Outlier Factor (LOF)


In [ ]:
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
y_pred_lof = lof.fit_predict(X_scaled)
y_pred_lof = (y_pred_lof == -1).astype(int)
print('=== Local Outlier Factor ===')
print(classification_report(y, y_pred_lof, target_names=['Normal', 'Anomaly']))
cm = confusion_matrix(y, y_pred_lof)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title('Confusion Matrix - LOF')
plt.show()


## 6. Perbandingan & Visualisasi


In [ ]:
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_pred_if, cmap='coolwarm', edgecolor='k', alpha=0.6)
plt.title('Isolation Forest - Hasil Deteksi')
plt.xlabel(X.columns[0])
plt.ylabel(X.columns[1])
plt.subplot(1, 2, 2)
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_pred_lof, cmap='coolwarm', edgecolor='k', alpha=0.6)
plt.title('LOF - Hasil Deteksi')
plt.xlabel(X.columns[0])
plt.ylabel(X.columns[1])
plt.tight_layout()
plt.show()


## 7. Pengaruh Parameter Contamination


In [ ]:
contaminations = [0.05, 0.1, 0.15, 0.2, 0.25]
scores = []
for c in contaminations:
    iso = IsolationForest(contamination=c, random_state=42)
    y_pred = (iso.fit_predict(X_scaled) == -1).astype(int)
    acc = np.mean(y_pred == y)
    scores.append(acc)
plt.plot(contaminations, scores, 'o-', linewidth=2)
plt.xlabel('Contamination')
plt.ylabel('Accuracy')
plt.title('Pengaruh Parameter Contamination')
plt.grid(True)
plt.show()


## 8. Kesimpulan
Isolation Forest dan LOF mampu mendeteksi anomali (fraud) pada data transaksi. Isolation Forest lebih cepat dan cocok untuk dataset besar. LOF lebih sensitif terhadap local density. Parameter contamination perlu disesuaikan dengan proporsi anomali aktual.
